In [ ]:
import cell2cell as c2c
import liana as li

import pandas as pd

import decoupler as dc

import matplotlib.pyplot as plt
import plotnine as p9
import seaborn as sns

%matplotlib inline

In [ ]:
import logging
logging.disable(logging.WARNING) # This is to avoid too many logs because of GSEApy

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#working env
import os
os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()
data_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/quickstart/'
output_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/'
c2c.io.directories.create_directory(data_folder)
c2c.io.directories.create_directory(output_folder)

In [ ]:
from matplotlib.pyplot import rc_context
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
factors = c2c.io.load_tensor_factors(output_folder + 'Loadings.xlsx')

In [ ]:
lr_pairs = li.resource.select_resource('mouseconsensus')
lr_pairs.head()

In [ ]:
lr_list = ['^'.join(row) for idx, row in lr_pairs.iterrows()] 

In [ ]:
organism = 'mouse' # For the COVID-19 data analyzed with LIANA+Tensor-cell2cell
pathwaydb = 'GOBP' # KEGG Pathways

In [ ]:
lr_set = c2c.external.generate_lr_geneset(lr_list,
                                          complex_sep='_',
                                          lr_sep='^',
                                          organism=organism,
                                          pathwaydb=pathwaydb,
                                          readable_name=True,
                                          output_folder=output_folder
                                          )

In [ ]:
lr_loadings = factors['Ligand-Receptor Pairs']

In [ ]:
%%time
pvals, scores, gsea_df = c2c.external.run_gsea(loadings=lr_loadings, 
                                               lr_set=lr_set,
                                               output_folder=output_folder,
                                               weight=1,
                                               min_size=15,
                                               permutations=999,
                                               processes=6,
                                               random_state=6,
                                               significance_threshold=1,
                                              )

In [ ]:
with sns.axes_style("darkgrid"):
    dotplot = c2c.plotting.pval_plot.generate_dot_plot(pval_df=pvals,
                                                       score_df=scores,
                                                       significance=0.1,
                                                       xlabel='',
                                                       ylabel='{} Annotations'.format(pathwaydb),
                                                       cbar_title='NES',
                                                       cmap='PuOr',
                                                       figsize=(5, 12),
                                                       label_size=20,
                                                       title_size=20,
                                                       tick_size=12,
                                                       filename=output_folder + '/GSEA-Dotplot.pdf'
                                                      )

In [ ]:
gsea_df.loc[(gsea_df['Adj. P-value'] < 0.05) & (gsea_df['NES'] > 0.)]

In [ ]:
gsea_df.loc[(gsea_df['Adj. P-value'] < 0.05) & (gsea_df['NES'] < 0.)]

In [ ]:
net = dc.get_progeny(organism='mouse', top=5000)

In [ ]:
lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^")
lr_progeny.head()

In [ ]:
estimate, pvals =  dc.run_mlm(lr_loadings.transpose(), 
                              lr_progeny, 
                              source="source", 
                              target="interaction", 
                              min_n=5,

                              use_raw=False)

In [ ]:
estimate
estimate.drop(columns=['Androgen', 'Estrogen'], inplace=True)

In [ ]:
sns.clustermap(estimate, xticklabels=estimate.columns, cmap='coolwarm', z_score=4, figsize=(4,4))
plt.savefig(output_folder + '/PROGENy.pdf', dpi=300, bbox_inches='tight')

In [ ]:
selected_factor = 'Factor 4'

dc.plot_barplot(estimate,
                selected_factor,
                vertical=True, 
                cmap='coolwarm', 
                figsize=(4, 4),
                save=output_folder + '/PROGENy-{}.pdf'.format(selected_factor.replace(' ', '-')))

In [ ]:
pathway = 'JAK-STAT'
# loadings to long format
lr_loadings_long = lr_loadings.reset_index().melt(id_vars='index', var_name="Factor", value_name="Loadings").rename(columns={'index':'interaction'})
# join progeny weights and keep only NFkB pathway
lr_loadings_long = lr_loadings_long.merge(lr_progeny, on='interaction').query("source == '{}' and Factor == '{}'".format(pathway, selected_factor))
# add sign to the weights
lr_loadings_long['sign'] = lr_loadings_long['weight'].apply(lambda x: 'positive' if x > 0 else 'negative')
# keep only relevant interactions for labels
lr_loadings_long['relevant_interactions'] = lr_loadings_long.apply(lambda x: x['interaction'] if (x['Loadings'] > 0.025) and (x['weight'] > 2.5) else None, axis=1)
lr_loadings_long.head()

In [ ]:
(p9.ggplot(lr_loadings_long,
           p9.aes(x='weight', y='Loadings')) +
 p9.geom_smooth(method='lm') + 
 p9.geom_point(p9.aes(colour='sign')) +
 p9.theme_bw() +
 p9.theme(legend_position='none') +
 p9.labs(title="{} | {}".format(pathway, selected_factor), x="PROGENy Weights", y="Loadings") +
 p9.scale_colour_manual(values=["royalblue", "red"]) +
 p9.geom_label(p9.aes(label='relevant_interactions'), size=8, nudge_y=0.01, nudge_x=0.01) + 
 p9.xlim(-1, 8.5)
 )